In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
import xgboost as xgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# Load merged dataset with Pinnacle spreads
df = pd.read_csv("nba_games_with_pinnacle_spreads.csv", low_memory=False)

# Quick preview
print(df.shape)
print(df.columns)
print(df.head())


(14894, 26)
Index(['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID',
       'VISITOR_TEAM_ID', 'SEASON', 'TEAM_ID_home', 'PTS_home', 'FG_PCT_home',
       'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'TEAM_ID_away',
       'PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away',
       'REB_away', 'HOME_TEAM_WINS', 'book_name', 'spread1', 'spread2',
       'price1', 'price2'],
      dtype='object')
  GAME_DATE_EST   GAME_ID GAME_STATUS_TEXT  HOME_TEAM_ID  VISITOR_TEAM_ID  \
0     11/1/2006  20600003            Final    1610612755       1610612737   
1     11/1/2006  20600004            Final    1610612766       1610612754   
2     11/1/2006  20600005            Final    1610612753       1610612741   
3     11/1/2006  20600006            Final    1610612738       1610612740   
4     11/1/2006  20600007            Final    1610612751       1610612761   

   SEASON  TEAM_ID_home  PTS_home  FG_PCT_home  FT_PCT_home  ...  FT_PCT_away  \
0    2006    1610

In [4]:
# Ensure game_id is clean
df["GAME_ID"] = df["GAME_ID"].astype(str).str.strip()

# Convert date column
df["GAME_DATE_EST"] = pd.to_datetime(df["GAME_DATE_EST"], errors="coerce")

# Drop rows with missing key data
df = df.dropna(subset=["PTS_home","PTS_away","spread1","spread2","price1","price2"])
# Quick preview
print(df.shape)
print(df.columns)
print(df.head())


(14894, 26)
Index(['GAME_DATE_EST', 'GAME_ID', 'GAME_STATUS_TEXT', 'HOME_TEAM_ID',
       'VISITOR_TEAM_ID', 'SEASON', 'TEAM_ID_home', 'PTS_home', 'FG_PCT_home',
       'FT_PCT_home', 'FG3_PCT_home', 'AST_home', 'REB_home', 'TEAM_ID_away',
       'PTS_away', 'FG_PCT_away', 'FT_PCT_away', 'FG3_PCT_away', 'AST_away',
       'REB_away', 'HOME_TEAM_WINS', 'book_name', 'spread1', 'spread2',
       'price1', 'price2'],
      dtype='object')
  GAME_DATE_EST   GAME_ID GAME_STATUS_TEXT  HOME_TEAM_ID  VISITOR_TEAM_ID  \
0    2006-11-01  20600003            Final    1610612755       1610612737   
1    2006-11-01  20600004            Final    1610612766       1610612754   
2    2006-11-01  20600005            Final    1610612753       1610612741   
3    2006-11-01  20600006            Final    1610612738       1610612740   
4    2006-11-01  20600007            Final    1610612751       1610612761   

   SEASON  TEAM_ID_home  PTS_home  FG_PCT_home  FT_PCT_home  ...  FT_PCT_away  \
0    2006    1610

In [5]:
# Point differential
df["point_diff"] = df["PTS_home"] - df["PTS_away"]

# Covered spread logic
df["home_covered"] = (df["point_diff"] + df["spread1"] > 0).astype(int)
df["away_covered"] = ((-df["point_diff"]) + df["spread2"] > 0).astype(int)

# Unified target column
df["covered_spread"] = df.apply(
    lambda row: row["home_covered"] if row["HOME_TEAM_ID"] == row["TEAM_ID_home"] else row["away_covered"],
    axis=1
)


In [6]:
# Basic features
df["is_home"] = (df["TEAM_ID_home"] == df["HOME_TEAM_ID"]).astype(int)

# Select features for modeling
features = ["is_home", "spread1", "spread2", "price1", "price2"]
X = df[features].fillna(0)
y = df["covered_spread"]


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [8]:

# Train Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

# Predictions
log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:,1]  # probability of covering spread

# Store probabilities in DataFrame
df.loc[X_test.index, "log_model_prob"] = log_prob


In [9]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

# Store probabilities in DataFrame
df.loc[X_test.index, "rf_model_prob"] = rf_prob

In [10]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="logloss"  
)
xgb_model.fit(X_train, y_train)

# Predictions
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:,1]

# Store probabilities in DataFrame
df.loc[X_test.index, "xgb_model_prob"] = xgb_prob

In [11]:
def evaluate_model(name, y_true, y_pred, y_prob):
    print(f"\n{name} Results")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Log Loss:", log_loss(y_true, y_prob))
    print("ROC-AUC:", roc_auc_score(y_true, y_prob))

evaluate_model("Logistic Regression", y_test, log_pred, log_prob)
evaluate_model("Random Forest", y_test, rf_pred, rf_prob)
evaluate_model("XGBoost", y_test, xgb_pred, xgb_prob)



Logistic Regression Results
Accuracy: 0.7912051023833501
Log Loss: 0.4424903824464467
ROC-AUC: 0.8623744465898231

Random Forest Results
Accuracy: 0.7891910036925143
Log Loss: 0.4520854092259147
ROC-AUC: 0.8538099974645526

XGBoost Results
Accuracy: 0.7912051023833501
Log Loss: 0.4456173111894822
ROC-AUC: 0.8575439315039105


In [12]:
results_summary = pd.DataFrame({
    "Model": ["Logistic Regression","Random Forest","XGBoost"],
    "Log Loss": [
        log_loss(y_test, log_prob),
        log_loss(y_test, rf_prob),
        log_loss(y_test, xgb_prob)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, log_prob),
        roc_auc_score(y_test, rf_prob),
        roc_auc_score(y_test, xgb_prob)
    ]
})

print("\nModel Comparison Summary:")
print(results_summary)



Model Comparison Summary:
                 Model  Log Loss   ROC-AUC
0  Logistic Regression  0.442490  0.862374
1        Random Forest  0.452085  0.853810
2              XGBoost  0.445617  0.857544


In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Target: point differential
y_reg = df["point_diff"]

X_reg = df[["is_home", "spread1", "spread2", "price1", "price2"]].fillna(0)

# Train/test split
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)
lin_pred = lin_reg.predict(X_test_reg)

print("Linear Regression Results")
print("MSE:", mean_squared_error(y_test_reg, lin_pred))
print("R²:", r2_score(y_test_reg, lin_pred))


Linear Regression Results
MSE: 137.66419535922878
R²: 0.22071377535978842


In [14]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting
gb_reg = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=5, random_state=42
)
gb_reg.fit(X_train_reg, y_train_reg)
gb_pred = gb_reg.predict(X_test_reg)

print("\nGradient Boosting Results")
print("MSE:", mean_squared_error(y_test_reg, gb_pred))
print("R²:", r2_score(y_test_reg, gb_pred))



Gradient Boosting Results
MSE: 140.58317332143642
R²: 0.20419009387499132


In [ ]:
# Convert American odds to decimal
def american_to_decimal(odds):
    if odds > 0:
        return (odds / 100) + 1
    else:
        return (100 / abs(odds)) + 1

df["decimal_price1"] = df["price1"].apply(american_to_decimal)
df["decimal_price2"] = df["price2"].apply(american_to_decimal)

# Example strategy: bet 1 unit on model's predicted side
df["bet_return"] = np.where(
    df["covered_spread"] == 1,  # if model predicted correctly
    df["decimal_price1"] - 1,   # profit from winning bet
    -1                          # loss of 1 unit
)

# Cumulative ROI over time
df["cumulative_roi"] = df["bet_return"].cumsum()

# Plot ROI curve
plt.figure(figsize=(12,6))
sns.lineplot(x=df["GAME_DATE_EST"], y=df["cumulative_roi"])
plt.title("Simulated ROI Over Time (Spread Strategy)")
plt.xlabel("Date")
plt.ylabel("Cumulative ROI (Units)")
plt.savefig("roi_curve.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Strategy 2: Threshold Betting 
# Convert Pinnacle odds to implied probabilities
df["implied_prob1"] = 1 / df["decimal_price1"]
df["implied_prob2"] = 1 / df["decimal_price2"]
df["model_prob"] = df["rf_model_prob"]  # can be changed to diff. model_probs

# Define threshold (e.g., 5% edge)
threshold = 0.05

# Bet only when model probability exceeds Pinnacle implied probability by threshold
df["bet_return_threshold"] = np.where(
    (df["model_prob"] - df["implied_prob1"]) > threshold,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0  # no bet placed
)

# Cumulative ROI
df["cumulative_roi_threshold"] = df["bet_return_threshold"].cumsum()

# Plot ROI curve
plt.figure(figsize=(12,6))
plt.plot(df["GAME_DATE_EST"], df["cumulative_roi_threshold"], label="Threshold Betting")
plt.title("ROI Curve: Threshold Betting Strategy")
plt.xlabel("Date")
plt.ylabel("Cumulative ROI (Units)")
plt.legend()
plt.savefig("roi_curve_threshold.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Strategy 3a: Favorites Only 
df["bet_return_favorites"] = np.where(
    df["spread1"] < 0,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0  # no bet placed
)

df["cumulative_roi_favorites"] = df["bet_return_favorites"].cumsum()

# Plot ROI curve
plt.figure(figsize=(12,6))
plt.plot(df["GAME_DATE_EST"], df["cumulative_roi_favorites"], label="Favorites Only")
plt.title("ROI Curve: Favorites Betting Strategy")
plt.xlabel("Date")
plt.ylabel("Cumulative ROI (Units)")
plt.legend()
plt.show()
plt.savefig("roi_curve_favorites.png", dpi=300, bbox_inches="tight")

In [ ]:
# Strategy 3b: Underdogs Only
df["bet_return_underdogs"] = np.where(
    df["spread1"] > 0,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0  # no bet placed
)

df["cumulative_roi_underdogs"] = df["bet_return_underdogs"].cumsum()

# Plot ROI curve
plt.figure(figsize=(12,6))
plt.plot(df["GAME_DATE_EST"], df["cumulative_roi_underdogs"], label="Underdogs Only")
plt.title("ROI Curve: Underdogs Betting Strategy")
plt.xlabel("Date")
plt.ylabel("Cumulative ROI (Units)")
plt.legend()
plt.savefig("roi_curve_underdogs.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ROI Comparison: Threshold Betting Across Models

# Define threshold
threshold = 0.05

# Logistic Regression strategy
df["log_threshold_return"] = np.where(
    (df["log_model_prob"] - df["implied_prob1"]) > threshold,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0
)
df["log_threshold_roi"] = df["log_threshold_return"].cumsum()

# Random Forest strategy
df["rf_threshold_return"] = np.where(
    (df["rf_model_prob"] - df["implied_prob1"]) > threshold,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0
)
df["rf_threshold_roi"] = df["rf_threshold_return"].cumsum()

# XGBoost strategy
df["xgb_threshold_return"] = np.where(
    (df["xgb_model_prob"] - df["implied_prob1"]) > threshold,
    np.where(df["covered_spread"] == 1, df["decimal_price1"] - 1, -1),
    0
)
df["xgb_threshold_roi"] = df["xgb_threshold_return"].cumsum()

# --- Plot comparison ---
plt.figure(figsize=(12,6))
plt.plot(df["GAME_DATE_EST"], df["log_threshold_roi"], label="Logistic Regression")
plt.plot(df["GAME_DATE_EST"], df["rf_threshold_roi"], label="Random Forest")
plt.plot(df["GAME_DATE_EST"], df["xgb_threshold_roi"], label="XGBoost")

plt.title("ROI Comparison: Threshold Betting Strategy Across Models")
plt.xlabel("Date")
plt.ylabel("Cumulative ROI (Units)")
plt.legend()
plt.savefig("roi_comparison_threshold_models.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Season-by-Season ROI Breakdown for Threshold Betting

# Group by season and sum returns for each model
season_roi_log = df.groupby("SEASON")["log_threshold_return"].sum()
season_roi_rf  = df.groupby("SEASON")["rf_threshold_return"].sum()
season_roi_xgb = df.groupby("SEASON")["xgb_threshold_return"].sum()

# Combine into one DataFrame
season_roi_table = pd.DataFrame({
    "Logistic Regression ROI": season_roi_log,
    "Random Forest ROI": season_roi_rf,
    "XGBoost ROI": season_roi_xgb
})

#plot as grouped bar chart
season_roi_table.plot(kind="bar", figsize=(12,6))
plt.title("Season-by-Season ROI Comparison (Threshold Betting)")
plt.xlabel("Season")
plt.ylabel("Total ROI (Units)")
plt.legend()
plt.savefig("season_roi_threshold_models.png", dpi=300, bbox_inches="tight")
plt.show()
